# phonex Indexer — GPU Acceleration on Colab

**Goal**: Use Colab's GPU to index your SOURCE CODE, then download the vector database to your machine.

**What you need**:
- Your **phonex project folder** (with `index_codebase.py`, `requirements.txt`, etc.)
- The **source code you want to index** (e.g., dotnet aspnetcore files)

**Before running**:
1. Go to `Runtime > Change runtime type > GPU` and select a GPU (T4 or better)
2. **ZIP your phonex folder** (with index_codebase.py inside it) and upload here
3. Run cells top-to-bottom

The Chroma vector index will be saved locally in Colab and ready to download.

In [ ]:
import torch
print("Python 3.x")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️  No GPU detected. Please enable GPU in Runtime > Change runtime type")

## 2. Upload Your Phonex Project

**To upload**: Your entire `phonex` folder (the one containing `index_codebase.py`, `server.py`, etc)

**How**:
1. On your local machine, ZIP your phonex folder
2. Run the cell below and select the ZIP file to upload
3. It will auto-extract

**Where is your phonex folder?** Typically at `D:\Gitrnd\phonex\`

In [ ]:
from google.colab import files
import os

print("Step 1: Upload your phonex.zip (or phonex_withsrc.zip)")
print("This should contain:")
print("  ✓ index_codebase.py")
print("  ✓ server.py")
print("  ✓ requirements.txt")
print("  ✓ Your source code (or a link to it)")
print("━" * 60)
print("\nSelect your ZIP file from your local machine:")

uploaded = files.upload()
for filename in uploaded.keys():
    print(f"\n✓ Uploaded: {filename}")
    if filename.endswith('.zip'):
        !unzip -q {filename} -d /content/
        !rm {filename}
        print(f"✓ Extracted to /content/")
    elif filename.endswith('.tar.gz'):
        !tar -xzf {filename} -C /content/
        !rm {filename}
        print(f"✓ Extracted to /content/")

print("\nContents of /content/:")
!find /content/ -maxdepth 2 -type f -name "*.py" | head -20

Step 1: Upload your phonex.zip (or phonex_withsrc.zip)
This should contain:
  ✓ index_codebase.py
  ✓ server.py
  ✓ requirements.txt
  ✓ Your source code (or a link to it)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Select your ZIP file from your local machine:


## 3. Install Dependencies

Install all required packages for indexing with GPU acceleration.

In [ ]:
!pip install -q llama-index chromadb sentence-transformers torch
!pip install -q llama-index-vector-stores-chroma
print("✓ All dependencies installed")

## 4. Run the Indexer

Find and run `index_codebase.py` (GPU will be used automatically for embeddings)

## 3b. Configure Source Code Path (Optional)

If your source code is in a different location, update it here before running the indexer:

In [ ]:
import os

# Find the uploaded phonex folder
phonex_path = None
for item in os.listdir('/content/'):
    if os.path.isdir(f'/content/{item}') and item.startswith('phonex'):
        phonex_path = f'/content/{item}'
        break

if not phonex_path:
    # fallback
    phonex_path = '/content/phonex'

print(f"Detected phonex at: {phonex_path}")
print(f"Files in {phonex_path}:")
if os.path.exists(phonex_path):
    !ls -la {phonex_path} | head -15
else:
    print("  (not found yet)")

print("\n" + "="*60)
print("Where is your SOURCE CODE to index?")
print("  /content/aspnetcore  (if you uploaded it)")
print("  /content/src  (or wherever you uploaded it)")
print("="*60)
print("\nIf your source code is elsewhere, update SOURCE_CODE_PATH below:")

# You can update this path if needed:
SOURCE_CODE_PATH = "/content/aspnetcore"  # Change this if different

print(f"\nUsing source code path: {SOURCE_CODE_PATH}")
print(f"Path exists? {os.path.exists(SOURCE_CODE_PATH)}")

if not os.path.exists(SOURCE_CODE_PATH):
    print("\n⚠️  Path not found. You may need to:")
    print("  1. Upload your source code separately, OR")
    print("  2. Update SOURCE_CODE_PATH above")

In [ ]:
import os
import subprocess

# Find index_codebase.py
search_paths = [
    '/content/phonex/index_codebase.py',
    '/content/index_codebase.py',
]

for p in search_paths:
    if os.path.exists(p):
        print(f"Found indexer at: {p}")
        print("=" * 70)
        subprocess.run(['python', p], check=False)
        break
else:
    print("ERROR: Could not find index_codebase.py")
    print("Expected at one of:")
    for p in search_paths:
        print(f"  - {p}")
    print("\nUpload the phonex folder and try again.")

## 5. Download the Vector Database

After indexing completes, zip and download the Chroma index.

In [ ]:
import os
import shutil
from google.colab import files

# Find the index folder (usually /tmp/phonex_index created by the script)
index_paths = [
    '/tmp/phonex_index',
    '/content/phonex_index',
    './phonex_index'
]

index_dir = None
for p in index_paths:
    if os.path.exists(p):
        index_dir = p
        break

if index_dir:
    print(f"Found index at: {index_dir}")
    print(f"Index size: {sum(os.path.getsize(os.path.join(dirpath, filename)) for dirpath, dirnames, filenames in os.walk(index_dir) for filename in filenames) / (1024**2):.2f} MB")
    
    # Zip it
    zip_path = '/tmp/phonex_index'
    shutil.make_archive(zip_path, 'zip', index_dir)
    print(f"\n✓ Zipped to {zip_path}.zip")
    
    # Download
    print("Downloading now...")
    files.download(f'{zip_path}.zip')
else:
    print("Could not find index folder at:")
    for p in index_paths:
        print(f"  - {p}")
    print("\nCheck that indexing completed successfully.")

## Done!

Your vector database (Chroma index) is now downloaded to your local machine.

### Next steps:
1. **Extract the ZIP**: Unzip `phonex_index.zip` on your local machine
2. **Use it locally**: Point your `server.py` or query script to the extracted index folder
3. **Queries will be fast**: Since embeddings are already computed on GPU, queries will be instant on your local machine

### If you want to use it in your local environment:
Update the `INDEX_STORAGE_DIR` in your local `index_codebase.py` to point to the extracted folder, or update `server.py` to load the Chroma index from the downloaded path.